In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root

session = get_active_session()
root = Root(session)

print(root)
print(session.sql("SELECT CURRENT_USER()").collect())

In [ ]:
from snowflake.core.database import Database
from snowflake.core import CreateMode

database = root.databases.create(
  Database(name="PYTHON_API_DB"),
  mode=CreateMode.or_replace
)

In [ ]:
from snowflake.core.schema import Schema

schema = database.schemas.create(
  Schema(name="PYTHON_API_SCHEMA"),
  mode=CreateMode.or_replace,
)

In [ ]:
from snowflake.core.table import Table, TableColumn

table = schema.tables.create(
  Table(
    name="PYTHON_API_TABLE",
    columns=[
      TableColumn(
        name="TEMPERATURE",
        datatype="int",
        nullable=False,
      ),
      TableColumn(
        name="LOCATION",
        datatype="string",
      ),
    ],
  ),
  mode=CreateMode.or_replace,
)

In [ ]:
table_details = table.fetch()
table_details.to_dict()

In [ ]:
from snowflake.core.table import PrimaryKey

# Remove existing 'elevation' column if present to avoid duplicates on re-run
table_details.columns = [c for c in table_details.columns if c.name != "elevation"]

table_details.columns.append(
    TableColumn(
      name="elevation",
      datatype="int",
      nullable=False,
      constraints=[PrimaryKey()],
    )
)
table.create_or_alter(table_details)
table.fetch().to_dict()

In [ ]:
from snowflake.core.warehouse import Warehouse

warehouses = root.warehouses

python_api_wh = Warehouse(
    name="PYTHON_API_WH",
    warehouse_size="SMALL",
    auto_suspend=500,
)

warehouse = warehouses.create(python_api_wh, mode=CreateMode.or_replace)

In [ ]:
warehouse_details = warehouse.fetch()
warehouse_details.to_dict()

In [ ]:
warehouse_list = warehouses.iter(like="PYTHON_API_WH")
result = next(warehouse_list)
result.to_dict()

In [ ]:
warehouse = root.warehouses.create(Warehouse(
    name="PYTHON_API_WH",
    warehouse_size="LARGE",
    auto_suspend=500,
), mode=CreateMode.or_replace)

warehouse.fetch().size

In [ ]:
warehouse.drop()